# Libraries

In [1]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from IPython.display import display

sns.set_theme(style="whitegrid")

ROOT = Path.cwd().resolve()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

PREDICTION_PATH = ROOT / "outputs" / "evaluation" / "layoutlmv3_spdocvqa" / "error_cases.json"

print("Project root:", ROOT)
print("Prediction file:", PREDICTION_PATH.exists())

Project root: C:\Users\firma\Documents\LOCKIN\SP-DOCVQA
Prediction file: False


# Data Information

In [2]:
if not PREDICTION_PATH.exists():
    raise FileNotFoundError(
        "Hasil analisis belum ditemukan. "
        "Jalankan python scripts/analyze_errors.py terlebih dahulu."
    )

with PREDICTION_PATH.open("r", encoding="utf-8") as file:
    predictions = json.load(file)

df = pd.DataFrame(predictions)

print("Jumlah data:", len(df))
display(df.head())

FileNotFoundError: Hasil analisis belum ditemukan. Jalankan python scripts/analyze_errors.py terlebih dahulu.

# Evaluation

## Statistik metrik

In [ ]:
metric_columns = ["exact_match", "anls", "iou"]
display(df[metric_columns].describe().round(4))

## Rata-rata metrik

In [ ]:
average_metrics = df[metric_columns].mean()

display(average_metrics.round(4))

average_metrics.plot(
    kind="bar",
    color=["#4472C4", "#70AD47", "#ED7D31"],
    figsize=(8, 5)
)

plt.ylim(0, 1)
plt.title("Rata-rata Metrik Evaluasi")
plt.xlabel("Metrik")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.show()

## Distribusi kategori kesalahan

In [ ]:
category_counts = df["error_category"].value_counts()

display(category_counts)

category_counts.sort_values().plot(
    kind="barh",
    figsize=(10, 5),
    color="#4472C4"
)

plt.title("Distribusi Kategori Kesalahan")
plt.xlabel("Jumlah Sampel")
plt.ylabel("Kategori")
plt.show()

## Hubungan ANLS dengan IoU

In [ ]:
plt.figure(figsize=(8, 6))

sns.scatterplot(
    data=df,
    x="anls",
    y="iou",
    hue="error_category",
    alpha=0.7
)

plt.title("Hubungan Akurasi Jawaban dan Lokasi")
plt.xlabel("ANLS")
plt.ylabel("IoU")
plt.show()

## Hasil berdasarkan question type

In [ ]:
question_type_result = (
    df.groupby("question_type")[metric_columns]
    .agg(["mean", "count"])
    .sort_values(("anls", "mean"), ascending=False)
)

display(question_type_result.round(4))

## Kesalahan terburuk

In [ ]:
worst_prediction_columns = [
    "question",
    "ground_truth_answers",
    "predicted_answer",
    "anls",
    "iou",
    "error_category"
]

worst_predictions = (
    df.sort_values(
        by=["anls", "iou"],
        ascending=[True, True]
    )[worst_prediction_columns]
    .head(20)
)

display(worst_predictions)